# Digital Signals Theory

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import soundfile as sf
#import warnings

from IPython.display import display, Audio
#from IPython.display import HTML
#from ipywidgets import Output, GridspecLayout

# Chapter 11: Infinite impulse response filters

## 11.1 Feedback filters

### 11.1.1. Example: exponential moving average

> As an introduction to feedback filters, let’s imagine the following process for computing a moving average of a signal $x[n]$, resulting in output signal $y[n]$.


\begin{flalign}
\qquad {\color{#984ea3}y[n]} &= {\color{#e41a1c}\frac{1}{2}} \cdot {\color{#377eb8}x[n]} + {\color{#4daf4a}\frac{1}{2}} \cdot {\color{#984ea3}y[n-1]} & \text{(11.1)}  \\
\end{flalign}

#### Key points

* This is a **feedback** process since ${\color{#984ea3}y[n]}$ depend on both **feed-forward** process ${\color{#377eb8}x[n]}$ as well as **previous output** ${\color{#984ea3}y[n-1]}$.
* Convention is that ${\color{#984ea3}y[-1] = 0}$, so initial output value is given by ${\color{#984ea3}y[0]} = {\color{#e41a1c}\frac{1}{2}} \cdot {\color{#377eb8}x[0]}$.
* As with convolutional filters, we can analyze the behavior of (11.1) by sending an impulse signal ${\color{#377eb8}x = [1, 0, 0, \ldots]}$ into the system and seeing how it responds.

\begin{flalign}
\qquad {\color{#377eb8}x[0] = 1} \quad \Rightarrow \quad &{\color{#984ea3}y[0] = \frac{1}{2}} & \\
\qquad {\color{#377eb8}x[1] = 0} \quad \Rightarrow \quad &{\color{#984ea3}y[1] = \frac{1}{2} \cdot y[0] = \frac{1}{4}} & \\
\qquad {\color{#377eb8}x[2] = 0} \quad \Rightarrow \quad &{\color{#984ea3}y[2] = \frac{1}{2} \cdot y[1] = \frac{1}{8}} & \\
\qquad {\color{#377eb8}x[3] = 0} \quad \Rightarrow \quad &{\color{#984ea3}y[3] = \frac{1}{2} \cdot y[2] = \frac{1}{16}} & \\
\qquad     & \ldots & \\
\qquad     &{\color{#984ea3}y[n] = 2^{-(n+1)}} & \\
\end{flalign}

* _... and now we know where "exponential" comes from!_
* _Note that ${\color{#984ea3}y}$ will continue to be halved, never actually reaching zero but always approaching it, and that's where "infinite" comes from..._

### 11.1.2. IIR Filters

#### Definition 11.1 (Linear IIR filter)

The general formula for a linear IIR filter is:

\begin{flalign}
\qquad {\color{#4daf4a}a[0]} \cdot {\color{#984ea3}y[n]} &= \sum_{k=0}^{K-1} {\color{#e41a1c}b[k]} \cdot {\color{#377eb8}x[n-k]} - \sum_{k=1}^{K-1} {\color{#4daf4a}a[k]} \cdot {\color{#984ea3}y[n-k]} & \text{(11.2)}  \\
\end{flalign}

where we have:

* **feed-forward** parameters ${\color{#e41a1c}b = [b_0, b_1, b_2, \ldots b_{K-1}]}$
* **feed-back** parameters ${\color{#4daf4a}a = [a_0, a_1, a_2, \ldots a_{K-1}]}$
* in order to avoid self-dependency case ${\color{#984ea3}y[n-0] = y[n]}$, treat ${\color{#4daf4a}a[0]}$ as a special case where for all intents and purposes we assume ${\color{#4daf4a}a[0] = 1}$
* so the **feed-back** parameters are in practice assumed to be  ${\color{#4daf4a}a = [1, a_1, a_2, \ldots a_{K-1}]}$, and this why that 2nd summation is $\sum_{{\color{#4daf4a}k=1}}^{K-1}$


### 11.1.3: Special case 1: ${\color{#4daf4a}a[0] = 1}$

As long as **feed-back** parameters are such that ${\color{#4daf4a}a[0] = 1}$, we can rearrange 11.2 to yield:

\begin{flalign}
\qquad {\color{#984ea3}y[n]} &= {\color{#4daf4a}\frac{1}{a[0]}} \cdot \left( \sum_{k=0}^{K-1} {\color{#e41a1c}b[k]} \cdot {\color{#377eb8}x[n-k]} - \sum_{k=1}^{K-1} {\color{#4daf4a}a[k]} \cdot {\color{#984ea3}y[n-k]} \right) & \\
\qquad                       &= \sum_{k=0}^{K-1} {\color{#e41a1c}b[k]} \cdot {\color{#377eb8}x[n-k]} - \sum_{k=1}^{K-1} {\color{#4daf4a}a[k]} \cdot {\color{#984ea3}y[n-k]} & \\
\end{flalign}

Looking back at 11.1 and the example of the exponential moving average, we can see:
* **feed-forward** parameters ${\color{#e41a1c}b = [\frac{1}{2}]}$
* **feed-back** parameters ${\color{#4daf4a}a = [1, -\frac{1}{2}]}$

### 11.1.4: Special case 2: ${\color{#4daf4a}a[k>0] = 0}$

In the case where **feed-back** parameters are such that ${\color{#4daf4a}a[0] = 1}$ and ${\color{#4daf4a}a[k>0] = 0}$, 11.2 reduces to _standard convolution_:

\begin{flalign}
\qquad {\color{#984ea3}y[n]} &= \sum_{k=0}^{K-1} {\color{#e41a1c}b[k]} \cdot {\color{#377eb8}x[n-k]}  & \\
\end{flalign}

* this means that FIR filters are a _special case of IIR!_

### 11.1.5. Causal filters

* The IIR filters we will study define ${\color{#984ea3}y[n]}$ in terms of inputs ${\color{#377eb8}x[n]}$ and with previous outputs up to and including ${\color{#984ea3}y[n-1]}$.
* Also note that for IIR, ${\color{#984ea3}y[n]}$ does **not** depend on _future values_ like ${\color{#377eb8}x[n+1]}$ or ${\color{#984ea3}y[n+1]}$.
* Systems with this property are **causal systems**.

### 11.1.6. Why are feedbacks subtracted instead of added?

* ... wait until next chapter, when we will learn about z-Transforms.

### 11.1.7. Summary

* Feedback systems _can have an infinite impulse response_, even when the input is finite in length.
* We can always assume the first feedback coefficient ${\color{#4daf4a}a[0] = 1}$.
* IIR filters generalize FIR filters.
* The feed-back terms are subtracted, not added, to produce each output ${\color{#984ea3}y[n]}$.
* In general, **feed-forward** parameters ${\color{#e41a1c}b}$ and **feed-back** parameters ${\color{#4daf4a}a}$ need not be the same length, but we can always pad the shorter of the two with zeros to make them match.

## 11.2 Using IIR filters